# Phase 5: Reranking (Colab Version)

This notebook:
- Trains XGBoost reranker on Google Colab
- Saves model to Google Drive
- Download model to use locally

**Before running:**
1. Upload to Google Drive `wholesale-project/` folder:
   - `labels_train.parquet`
   - `labels_test.parquet`
   - (indices folder should already be there from Phase 3)

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q rank-bm25 sentence-transformers faiss-cpu xgboost pyarrow

In [ ]:
# Imports
import pandas as pd
import numpy as np
import pickle
import json
import time
import re
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from tqdm import tqdm

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss
import xgboost as xgb

print("✓ Imports successful")

In [ ]:
# Set paths
drive_path = '/content/drive/MyDrive/wholesale-project'
indices_dir = f'{drive_path}/indices'

print(f"Drive path: {drive_path}")
print(f"Indices dir: {indices_dir}")

## 2. Load Retriever (Inline)

In [ ]:
# Load BM25
print("Loading BM25...")
with open(f'{indices_dir}/bm25_index.pkl', 'rb') as f:
    bm25_data = pickle.load(f)
bm25 = bm25_data['bm25']
tokenized_corpus = bm25_data['tokenized_corpus']
print(f"✓ BM25 loaded: {len(tokenized_corpus):,} documents")

# Load FAISS
print("Loading FAISS...")
faiss_index = faiss.read_index(f'{indices_dir}/faiss_index.bin')
print(f"✓ FAISS loaded: {faiss_index.ntotal:,} vectors")

# Load product IDs
with open(f'{indices_dir}/product_ids.json', 'r') as f:
    product_ids = json.load(f)
print(f"✓ Product IDs loaded: {len(product_ids):,}")

# Load embedding model
print("Loading embedding model...")
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("✓ Embedding model loaded")

In [ ]:
# Simple retriever functions
def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

def search_bm25(query, top_k=100):
    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in top_indices:
        if scores[idx] > 0:
            results.append({'index': int(idx), 'product_id': product_ids[idx], 'score': float(scores[idx]), 'source': 'bm25'})
    return results

def search_faiss(query, top_k=100):
    query_emb = embedding_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_emb)
    scores, indices = faiss_index.search(query_emb, top_k)
    results = []
    for i, (idx, score) in enumerate(zip(indices[0], scores[0])):
        results.append({'index': int(idx), 'product_id': product_ids[idx], 'score': float(score), 'source': 'faiss'})
    return results

def search_hybrid(query, top_k=50, k=60):
    bm25_results = search_bm25(query, top_k=100)
    faiss_results = search_faiss(query, top_k=100)
    
    scores = {}
    sources = {}
    
    for rank, r in enumerate(bm25_results):
        idx = r['index']
        scores[idx] = scores.get(idx, 0) + 1.0 / (k + rank + 1)
        sources[idx] = sources.get(idx, []) + ['bm25']
    
    for rank, r in enumerate(faiss_results):
        idx = r['index']
        scores[idx] = scores.get(idx, 0) + 1.0 / (k + rank + 1)
        sources[idx] = sources.get(idx, []) + ['faiss']
    
    sorted_indices = sorted(scores.keys(), key=lambda x: scores[x], reverse=True)
    
    results = []
    for idx in sorted_indices[:top_k]:
        results.append({'index': idx, 'product_id': product_ids[idx], 'score': scores[idx], 'sources': sources[idx]})
    return results

print("✓ Retriever functions ready")

In [ ]:
# Quick test
test_results = search_hybrid("ceramic mugs", top_k=5)
print(f"Test search returned {len(test_results)} results")
for r in test_results[:3]:
    print(f"  - {r['product_id']} (score: {r['score']:.4f})")

## 3. Load Data

In [ ]:
# Load product data
df_products = pd.read_parquet(f'{drive_path}/products.parquet')

# Filter to indexed products
indexed_ids = set(product_ids)
df_products = df_products[df_products['product_id'].isin(indexed_ids)].reset_index(drop=True)

print(f"✓ Loaded {len(df_products):,} products")

In [ ]:
# Load labeled data
df_train_labels = pd.read_parquet(f'{drive_path}/labels_train.parquet')
df_test_labels = pd.read_parquet(f'{drive_path}/labels_test.parquet')

# Filter to indexed products
df_train_labels = df_train_labels[df_train_labels['product_id'].isin(indexed_ids)]
df_test_labels = df_test_labels[df_test_labels['product_id'].isin(indexed_ids)]

print(f"✓ Train labels: {len(df_train_labels):,} pairs ({df_train_labels['query'].nunique():,} queries)")
print(f"✓ Test labels: {len(df_test_labels):,} pairs ({df_test_labels['query'].nunique():,} queries)")

In [ ]:
# Check label distribution
print("\nLabel Distribution (Train):")
print(df_train_labels['esci_label'].value_counts())

## 4. Feature Extractor & Reranker Classes

In [ ]:
class FeatureExtractor:
    """Extract features for reranking."""
    
    def __init__(self, product_df):
        self.product_lookup = product_df.set_index('product_id').to_dict('index')
    
    def extract_features(self, query, results, query_analysis=None):
        features = []
        query_tokens = set(query.lower().split())
        is_wholesale = any(w in query.lower() for w in ['bulk', 'wholesale', 'pack', 'case'])
        
        for rank, result in enumerate(results):
            product = self.product_lookup.get(result['product_id'], {})
            title = str(product.get('product_title', '')).lower()
            brand = str(product.get('product_brand', '')).lower()
            title_tokens = set(title.split())
            
            feat = []
            feat.append(result.get('score', 0))  # hybrid score
            feat.append(1.0 / (rank + 1))  # rank score
            
            sources = result.get('sources', [])
            feat.append(1.0 if 'bm25' in sources else 0.0)
            feat.append(1.0 if 'faiss' in sources else 0.0)
            feat.append(1.0 if ('bm25' in sources and 'faiss' in sources) else 0.0)
            
            feat.append(min(len(title) / 200.0, 1.0))  # title length
            
            overlap = len(query_tokens & title_tokens)
            feat.append(overlap / max(len(query_tokens), 1))  # word overlap
            feat.append(overlap / max(len(query_tokens), 1))  # coverage
            
            feat.append(1.0 if brand and brand in query.lower() else 0.0)  # brand match
            
            has_bulk = any(w in title for w in ['bulk', 'pack', 'set of', 'wholesale', 'case'])
            feat.append(1.0 if (is_wholesale and has_bulk) else 0.0)  # wholesale match
            
            feat.append(1.0 if '$' in title or 'oz' in title else 0.0)  # price indicator
            
            completeness = sum([
                1 if product.get('product_brand') else 0,
                1 if product.get('product_description') else 0,
                1 if product.get('product_bullet_point') else 0,
            ]) / 3.0
            feat.append(completeness)
            
            features.append(feat)
        
        return np.array(features, dtype=np.float32)
    
    @property
    def feature_names(self):
        return ['hybrid_score', 'rank_score', 'has_bm25', 'has_faiss', 'has_both',
                'title_length', 'word_overlap', 'query_coverage', 'brand_match',
                'wholesale_match', 'has_price_indicator', 'product_completeness']

print("✓ FeatureExtractor defined")

In [ ]:
class Reranker:
    """XGBoost reranker."""
    
    def __init__(self, product_df=None, model_path=None):
        self.model = None
        self.feature_extractor = None
        if product_df is not None:
            self.feature_extractor = FeatureExtractor(product_df)
        if model_path and Path(model_path).exists():
            self.load(model_path)
    
    def train(self, train_data, val_data=None, params=None):
        if params is None:
            params = {
                'objective': 'rank:ndcg',
                'learning_rate': 0.1,
                'max_depth': 6,
                'n_estimators': 100,
                'subsample': 0.8,
                'colsample_bytree': 0.8,
                'random_state': 42,
                'n_jobs': -1
            }
        
        X_train, y_train, groups_train = self._prepare_data(train_data)
        print(f"Training data: {X_train.shape[0]} samples, {len(groups_train)} queries")
        
        self.model = xgb.XGBRanker(**params)
        
        if val_data:
            X_val, y_val, groups_val = self._prepare_data(val_data)
            self.model.fit(X_train, y_train, group=groups_train,
                          eval_set=[(X_val, y_val)], eval_group=[groups_val], verbose=True)
        else:
            self.model.fit(X_train, y_train, group=groups_train, verbose=True)
        
        print("✓ Reranker trained")
        self._print_feature_importance()
    
    def _prepare_data(self, data):
        X_list, y_list, groups = [], [], []
        for item in data:
            if len(item['results']) == 0:
                continue
            features = self.feature_extractor.extract_features(item['query'], item['results'])
            X_list.append(features)
            y_list.extend(item['labels'])
            groups.append(len(item['results']))
        return np.vstack(X_list), np.array(y_list, dtype=np.float32), groups
    
    def _print_feature_importance(self):
        if self.model is None:
            return
        importance = self.model.feature_importances_
        names = self.feature_extractor.feature_names
        print("\nFeature Importance:")
        print("-" * 40)
        for name, imp in sorted(zip(names, importance), key=lambda x: -x[1]):
            print(f"  {name}: {imp:.4f}")
    
    def rerank(self, query, results, top_k=None):
        if len(results) == 0 or self.model is None:
            return results[:top_k] if top_k else results
        features = self.feature_extractor.extract_features(query, results)
        scores = self.model.predict(features)
        sorted_indices = np.argsort(scores)[::-1]
        reranked = []
        for new_rank, old_idx in enumerate(sorted_indices):
            result = results[old_idx].copy()
            result['rerank_score'] = float(scores[old_idx])
            result['original_rank'] = old_idx + 1
            result['new_rank'] = new_rank + 1
            reranked.append(result)
        return reranked[:top_k] if top_k else reranked
    
    def save(self, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'wb') as f:
            pickle.dump({'model': self.model, 'feature_names': self.feature_extractor.feature_names}, f)
        print(f"✓ Model saved to {path}")
    
    def load(self, path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
        self.model = data['model']
        print(f"✓ Model loaded from {path}")

print("✓ Reranker defined")

## 5. Create Training Data

In [ ]:
def create_training_data(labels_df, max_queries=500, candidates_per_query=50):
    """Create training data from labels."""
    label_map = {'E': 3, 'S': 2, 'C': 1, 'I': 0}
    queries = labels_df['query'].unique()[:max_queries]
    
    training_data = []
    for query in tqdm(queries, desc="Creating data"):
        query_labels = labels_df[labels_df['query'] == query]
        label_dict = dict(zip(query_labels['product_id'], query_labels['esci_label']))
        
        results = search_hybrid(query, top_k=candidates_per_query)
        if len(results) == 0:
            continue
        
        labels = [label_map.get(label_dict.get(r['product_id'], 'I'), 0) for r in results]
        training_data.append({'query': query, 'results': results, 'labels': labels})
    
    return training_data

print("✓ Training data function defined")

In [ ]:
# Create training data
print("Creating training data...\n")
train_data = create_training_data(df_train_labels, max_queries=500, candidates_per_query=50)
print(f"\n✓ Created {len(train_data)} training queries")
print(f"  Total samples: {sum(len(d['labels']) for d in train_data):,}")

In [ ]:
# Create validation data
print("Creating validation data...\n")
val_data = create_training_data(df_test_labels, max_queries=200, candidates_per_query=50)
print(f"\n✓ Created {len(val_data)} validation queries")
print(f"  Total samples: {sum(len(d['labels']) for d in val_data):,}")

In [ ]:
# Check label distribution
all_labels = [l for d in train_data for l in d['labels']]
print("\nLabel distribution in training data:")
print(pd.Series(all_labels).value_counts().sort_index())
print("\n0=Irrelevant, 1=Complement, 2=Substitute, 3=Exact")

## 6. Train Reranker

In [ ]:
# Initialize reranker
reranker = Reranker(product_df=df_products)
print("✓ Reranker initialized")

In [ ]:
# Train
print("Training reranker...\n")
reranker.train(
    train_data=train_data,
    val_data=val_data,
    params={
        'objective': 'rank:ndcg',
        'learning_rate': 0.1,
        'max_depth': 6,
        'n_estimators': 100,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': 42,
        'n_jobs': -1
    }
)

## 7. Test Reranking

In [ ]:
# Product lookup for display
product_lookup = df_products.set_index('product_id').to_dict('index')

def display_comparison(query, top_k=5):
    hybrid_results = search_hybrid(query, top_k=50)
    reranked_results = reranker.rerank(query, hybrid_results, top_k=10)
    
    print(f"\nQuery: '{query}'")
    print("=" * 70)
    
    print("\nBefore Reranking (Hybrid):")
    for i, r in enumerate(hybrid_results[:top_k]):
        title = product_lookup.get(r['product_id'], {}).get('product_title', 'N/A')[:50]
        print(f"  {i+1}. {title}...")
    
    print("\nAfter Reranking:")
    for i, r in enumerate(reranked_results[:top_k]):
        title = product_lookup.get(r['product_id'], {}).get('product_title', 'N/A')[:50]
        orig = r.get('original_rank', '?')
        print(f"  {i+1}. (was #{orig}) {title}...")

In [ ]:
# Test queries
test_queries = [
    "ceramic mugs bulk",
    "nike running shoes",
    "organic candles lavender",
    "iphone case",
    "eco friendly bags wholesale"
]

for query in test_queries:
    display_comparison(query)

## 8. Save Model to Google Drive

In [ ]:
# Create models directory
models_dir = Path(f'{drive_path}/models')
models_dir.mkdir(parents=True, exist_ok=True)

# Save model
reranker.save(f'{models_dir}/reranker.pkl')

In [ ]:
# Verify it loads
reranker_test = Reranker(product_df=df_products, model_path=f'{models_dir}/reranker.pkl')
test_results = search_hybrid("ceramic mugs", top_k=10)
reranked = reranker_test.rerank("ceramic mugs", test_results, top_k=5)
print(f"✓ Model verified! Got {len(reranked)} results.")

## 9. Summary

In [ ]:
print("\n" + "=" * 60)
print("PHASE 5 COMPLETE (Colab)")
print("=" * 60)

print(f"""
Reranking Summary
-----------------
Model: XGBoost Ranker (LambdaMART-style)
Training queries: {len(train_data)}
Validation queries: {len(val_data)}

Model saved to Google Drive:
  {models_dir}/reranker.pkl

NEXT STEPS:
1. Go to Google Drive
2. Download 'models/reranker.pkl'
3. Put it in your local: data/models/reranker.pkl
4. Continue with Phase 6 locally!
""")